In [2]:
import pandas as pd

books=pd.read_csv('books_with_categories.csv')

books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction


In [30]:
# Use a pipeline as a high-level helper
from transformers import pipeline

classifier = pipeline("text-classification",
                model="j-hartmann/emotion-english-distilroberta-base",
                top_k=None)
classifier('I love this')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1182.39it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'joy', 'score': 0.9845667481422424},
  {'label': 'surprise', 'score': 0.0049272081814706326},
  {'label': 'sadness', 'score': 0.004531427752226591},
  {'label': 'neutral', 'score': 0.003475315636023879},
  {'label': 'anger', 'score': 0.0013875303557142615},
  {'label': 'disgust', 'score': 0.0007134040934033692},
  {'label': 'fear', 'score': 0.00039849060704000294}]]

In [37]:
len(books['description'][0])

1154

In [40]:
sentences = books['description'][0].split('.')
predictions = classifier(sentences)

In [54]:
sentences[5]

' Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world has to offer'

In [55]:
predictions[5]

[{'label': 'joy', 'score': 0.9327967166900635},
 {'label': 'disgust', 'score': 0.037718113511800766},
 {'label': 'neutral', 'score': 0.015891969203948975},
 {'label': 'sadness', 'score': 0.006444588769227266},
 {'label': 'anger', 'score': 0.005025044549256563},
 {'label': 'surprise', 'score': 0.0015812116907909513},
 {'label': 'fear', 'score': 0.0005423121037892997}]

In [84]:
import numpy as np

emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']
isbn=[]
emotion_scores={label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for predictions in predictions:
        sorted_predictions = sorted(predictions, key=lambda x: x['label'])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]['score'])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}
 

In [85]:
for i in range(10):
    isbn.append(books['isbn13'][i])
    sentences=books['description'][i].split('.')
    predictions=classifier(sentences)
    max_scores=calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [88]:
emotion_scores

{'anger': [np.float64(0.0641336515545845),
  np.float64(0.612618625164032),
  np.float64(0.0641336515545845),
  np.float64(0.3514834940433502),
  np.float64(0.08141222596168518),
  np.float64(0.2322252243757248),
  np.float64(0.5381842851638794),
  np.float64(0.0641336515545845),
  np.float64(0.30066969990730286),
  np.float64(0.0641336515545845)],
 'disgust': [np.float64(0.2735920548439026),
  np.float64(0.34828436374664307),
  np.float64(0.10400673747062683),
  np.float64(0.15072272717952728),
  np.float64(0.1844952255487442),
  np.float64(0.7271744608879089),
  np.float64(0.15585488080978394),
  np.float64(0.10400673747062683),
  np.float64(0.27948158979415894),
  np.float64(0.177927628159523)],
 'fear': [np.float64(0.9281686544418335),
  np.float64(0.9425276517868042),
  np.float64(0.9723208546638489),
  np.float64(0.3607065975666046),
  np.float64(0.09504332393407822),
  np.float64(0.051362842321395874),
  np.float64(0.7474281191825867),
  np.float64(0.4044967293739319),
  np.floa

In [103]:
from tqdm import tqdm

emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']
isbn=[]
emotion_scores={label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books['isbn13'][i])
    sentences=books['description'][i].split('.')
    predictions=classifier(sentences)
    max_scores=calculate_max_emotion_scores(predictions)
    for label in emotion_labels: 
        emotion_scores[label].append(max_scores[label])


100%|██████████| 5197/5197 [17:56<00:00,  4.83it/s]   


In [104]:
emotions_df=pd.DataFrame(emotion_scores)
emotions_df['isbn13']=isbn


In [105]:
books['isbn13'].head()

0    9780002005883
1    9780002261982
2    9780006178736
3    9780006280897
4    9780006280934
Name: isbn13, dtype: int64

In [107]:
emotions_df['isbn13'].head()

0    9780002005883
1    9780002261982
2    9780006178736
3    9780006280897
4    9780006280934
Name: isbn13, dtype: int64

In [108]:
books=pd.merge(books, emotions_df, on='isbn13')

In [109]:
books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.064134,0.273592,0.928169,0.932797,0.646217,0.967158,0.729603
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.612619,0.348284,0.942528,0.704422,0.887940,0.111690,0.252545
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,0.078766
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.351483,0.150723,0.360707,0.251881,0.732686,0.111690,0.078766
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.081412,0.184495,0.095043,0.040564,0.884390,0.475881,0.078766


In [111]:
books.to_csv('books_with_emotions.csv', index=False)